In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import scipy.sparse as sp
import os
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
TRAIN_FILE = "/kaggle/input/datasets/mantriaditya/gowalla/train.txt"
TEST_FILE  = "/kaggle/input/datasets/mantriaditya/gowalla/test.txt"
SAVE_DIR   = "/kaggle/working/lightgcn_single_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

In [4]:
EMBED_DIM  = 64
N_LAYERS   = 4
LR         = 1e-3
LAMBDA     = 1e-4
BATCH_SIZE = 1024
N_EPOCHS   = 100
TOPK       = 20
EVAL_EVERY = 5

In [5]:
def load_data(train_file, test_file):
    train_dict = defaultdict(set)
    test_dict  = defaultdict(set)

    with open(train_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            uid = int(parts[0])
            for iid in parts[1:]:
                train_dict[uid].add(int(iid))

    with open(test_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            uid = int(parts[0])
            for iid in parts[1:]:
                test_dict[uid].add(int(iid))

    all_users = set(train_dict.keys()) | set(test_dict.keys())
    all_items = set()

    for items in train_dict.values():
        all_items |= items
    for items in test_dict.values():
        all_items |= items

    n_users = max(all_users) + 1
    n_items = max(all_items) + 1
    return train_dict, test_dict, n_users, n_items


train_dict, test_dict, n_users, n_items = load_data(TRAIN_FILE, TEST_FILE)

n_train = sum(len(v) for v in train_dict.values())
n_test  = sum(len(v) for v in test_dict.values())

print(f"Users        : {n_users}")
print(f"Items        : {n_items}")
print(f"Train inters : {n_train}")
print(f"Test  inters : {n_test}")
print(f"Density      : {n_train / (n_users * n_items):.6f}")

Users        : 29858
Items        : 40981
Train inters : 810128
Test  inters : 217242
Density      : 0.000662


In [6]:
def build_norm_adj(train_dict, n_users, n_items):
    rows, cols = [], []

    for uid, items in train_dict.items():
        for iid in items:
            rows.append(uid)
            cols.append(iid + n_users)

    n       = n_users + n_items
    data    = np.ones(len(rows), dtype=np.float32)
    row_all = np.array(rows + cols, dtype=np.int32)
    col_all = np.array(cols + rows, dtype=np.int32)
    dat_all = np.concatenate([data, data])

    A = sp.csr_matrix((dat_all, (row_all, col_all)), shape=(n, n))

    deg        = np.array(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(deg > 0, deg ** -0.5, 0.0).astype(np.float32)
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    A_norm     = D_inv_sqrt @ A @ D_inv_sqrt

    A_coo    = A_norm.tocoo().astype(np.float32)
    indices  = torch.from_numpy(np.vstack([A_coo.row, A_coo.col])).long()
    values   = torch.from_numpy(A_coo.data)
    A_tensor = torch.sparse_coo_tensor(
        indices, values, torch.Size(A_coo.shape)
    ).coalesce().to(device)

    return A_tensor


A_norm = build_norm_adj(train_dict, n_users, n_items)
print(f"Adj matrix built: {n_users + n_items} x {n_users + n_items}")

Adj matrix built: 70839 x 70839


In [7]:
class LightGCNSingle(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, n_layers, A_norm):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.A_norm   = A_norm

        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self):
        E = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        all_layers = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.A_norm, E)
            all_layers.append(E)

        # LightGCN-single: use ONLY the final layer
        E_final = all_layers[-1]

        return E_final[:self.n_users], E_final[self.n_users:]

    def forward(self, users, pos_items, neg_items):
        u_emb, i_emb = self.propagate()
        u  = u_emb[users]
        pi = i_emb[pos_items]
        ni = i_emb[neg_items]
        return (u * pi).sum(-1), (u * ni).sum(-1)

    @torch.no_grad()
    def get_embeddings(self):
        return self.propagate()


model = LightGCNSingle(n_users, n_items, EMBED_DIM, N_LAYERS, A_norm).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

Trainable parameters: 4,533,696


In [8]:
def bpr_loss(pos_scores, neg_scores, users, pos_items, neg_items, model, lam):
    mf_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()

    u0  = model.user_emb(users)
    pi0 = model.item_emb(pos_items)
    ni0 = model.item_emb(neg_items)

    reg = (
        u0.norm(2).pow(2) +
        pi0.norm(2).pow(2) +
        ni0.norm(2).pow(2)
    ) / (2 * len(users))

    return mf_loss + lam * reg


print("Building training pair array...")
train_pairs = np.array(
    [(u, i) for u, items in train_dict.items() for i in items],
    dtype=np.int32
)
print(f"Total train pairs: {len(train_pairs):,}")

Building training pair array...
Total train pairs: 810,128


In [9]:
def sample_batch(train_pairs, train_dict, n_items, batch_size):
    idx       = np.random.randint(0, len(train_pairs), batch_size)
    users     = train_pairs[idx, 0]
    pos_items = train_pairs[idx, 1]
    neg_items = np.random.randint(0, n_items, batch_size)

    for j in range(batch_size):
        while neg_items[j] in train_dict[users[j]]:
            neg_items[j] = np.random.randint(0, n_items)

    return (
        torch.LongTensor(users).to(device),
        torch.LongTensor(pos_items).to(device),
        torch.LongTensor(neg_items).to(device)
    )


In [10]:
@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_users, n_items, k=20, batch=512):
    model.eval()
    u_emb, i_emb = model.get_embeddings()
    i_emb_T = i_emb.T

    total_recall = 0.0
    total_ndcg   = 0.0
    test_users   = [u for u in test_dict if len(test_dict[u]) > 0]

    for start in range(0, len(test_users), batch):
        batch_users = test_users[start:start + batch]
        u_vecs = u_emb[torch.LongTensor(batch_users).to(device)]
        scores = torch.mm(u_vecs, i_emb_T).cpu().numpy()

        for idx, user in enumerate(batch_users):
            for ti in train_dict.get(user, []):
                scores[idx][ti] = -np.inf

            top_k  = np.argsort(-scores[idx])[:k]
            gt_set = test_dict[user]

            hits = len(set(top_k) & gt_set)
            total_recall += hits / min(len(gt_set), k)

            dcg = sum(
                1.0 / np.log2(r + 2)
                for r, item in enumerate(top_k) if item in gt_set
            )
            idcg = sum(
                1.0 / np.log2(r + 2)
                for r in range(min(len(gt_set), k))
            )
            total_ndcg += dcg / idcg if idcg > 0 else 0.0

    n = len(test_users)
    return total_recall / n, total_ndcg / n

In [11]:
steps_per_epoch = max(1, len(train_pairs) // BATCH_SIZE)

train_losses   = []
recall_history = []
ndcg_history   = []
eval_epochs    = []
best_recall    = 0.0

print(f"LightGCN-single | K={N_LAYERS} | dim={EMBED_DIM} | epochs={N_EPOCHS} | steps/epoch={steps_per_epoch}\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    pbar = tqdm(
        range(steps_per_epoch),
        desc=f"Epoch {epoch:03d}/{N_EPOCHS}",
        leave=False,
        dynamic_ncols=True
    )

    for _ in pbar:
        users, pos_items, neg_items = sample_batch(
            train_pairs, train_dict, n_items, BATCH_SIZE
        )

        pos_scores, neg_scores = model(users, pos_items, neg_items)
        loss = bpr_loss(
            pos_scores, neg_scores,
            users, pos_items, neg_items, model, LAMBDA
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / steps_per_epoch
    train_losses.append(avg_loss)

    if epoch % EVAL_EVERY == 0 or epoch == 1:
        recall, ndcg = evaluate(
            model, train_dict, test_dict, n_users, n_items, k=TOPK
        )
        recall_history.append(recall)
        ndcg_history.append(ndcg)
        eval_epochs.append(epoch)

        flag = "  ← best" if recall > best_recall else ""
        print(
            f"Epoch {epoch:03d} | loss {avg_loss:.4f} | "
            f"Recall@{TOPK}: {recall:.4f} | NDCG@{TOPK}: {ndcg:.4f}{flag}"
        )

        if recall > best_recall:
            best_recall = recall
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'recall': recall,
                'ndcg': ndcg,
            }, os.path.join(SAVE_DIR, 'best_model.pt'))

    if epoch % 20 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'recall_history': recall_history,
            'ndcg_history': ndcg_history,
            'eval_epochs': eval_epochs,
        }, os.path.join(SAVE_DIR, f'ckpt_epoch{epoch:03d}.pt'))
        print(f"  [checkpoint saved @ epoch {epoch}]")


LightGCN-single | K=4 | dim=64 | epochs=100 | steps/epoch=791



Epoch 001 | loss 0.3210 | Recall@20: 0.0850 | NDCG@20: 0.0727  ← best


Epoch 005 | loss 0.1269 | Recall@20: 0.1038 | NDCG@20: 0.0896  ← best


Epoch 010 | loss 0.1066 | Recall@20: 0.1147 | NDCG@20: 0.0989  ← best


Epoch 015 | loss 0.0973 | Recall@20: 0.1219 | NDCG@20: 0.1048  ← best


...

Epoch 080 | loss 0.0796 | Recall@20: 0.1392 | NDCG@20: 0.1174  ← best
  [checkpoint saved @ epoch 80]


Epoch 085 | loss 0.0798 | Recall@20: 0.1394 | NDCG@20: 0.1174  ← best


Epoch 090 | loss 0.0793 | Recall@20: 0.1400 | NDCG@20: 0.1179  ← best


Epoch 095 | loss 0.0786 | Recall@20: 0.1411 | NDCG@20: 0.1186  ← best


Epoch 100 | loss 0.0784 | Recall@20: 0.1407 | NDCG@20: 0.1186
  [checkpoint saved @ epoch 100]


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'LightGCN-single | Gowalla | K={N_LAYERS} | dim={EMBED_DIM}',
    fontsize=14,
    fontweight='bold'
)

axes[0].plot(range(1, len(train_losses) + 1), train_losses,
             color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BPR Loss')
axes[0].set_title('Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(eval_epochs, recall_history,
             color='darkorange', marker='o', markersize=4, linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel(f'Recall@{TOPK}')
axes[1].set_title(f'Recall@{TOPK}')
axes[1].grid(alpha=0.3)

axes[2].plot(eval_epochs, ndcg_history,
             color='forestgreen', marker='s', markersize=4, linewidth=1.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel(f'NDCG@{TOPK}')
axes[2].set_title(f'NDCG@{TOPK}')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved → {plot_path}")


best_ckpt = torch.load(
    os.path.join(SAVE_DIR, 'best_model.pt'),
    weights_only=False
)
model.load_state_dict(best_ckpt['model_state_dict'])

final_recall, final_ndcg = evaluate(
    model, train_dict, test_dict, n_users, n_items, k=TOPK
)

results = {
    'dataset'        : 'Gowalla',
    'model'          : 'LightGCN-single',
    'n_users'        : n_users,
    'n_items'        : n_items,
    'n_train'        : int(n_train),
    'n_test'         : int(n_test),
    'embed_dim'      : EMBED_DIM,
    'n_layers'       : N_LAYERS,
    'lr'             : LR,
    'lambda'         : LAMBDA,
    'batch_size'     : BATCH_SIZE,
    'epochs_trained' : N_EPOCHS,
    'best_epoch'     : int(best_ckpt['epoch']),
    f'recall@{TOPK}' : round(final_recall, 4),
    f'ndcg@{TOPK}'   : round(final_ndcg, 4),
    'train_losses'   : train_losses,
    'eval_epochs'    : eval_epochs,
    'recall_history' : recall_history,
    'ndcg_history'   : ndcg_history,
}

with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("=" * 55)
print("FINAL RESULTS  (best checkpoint)")
print("=" * 55)
print(f"  Best Epoch    : {best_ckpt['epoch']}")
print(f"  Recall@{TOPK} : {final_recall:.4f}")
print(f"  NDCG@{TOPK}   : {final_ndcg:.4f}")
print("=" * 55)
print(f"\nSaved to {SAVE_DIR}:")
print("  best_model.pt  |  ckpt_epoch*.pt  |  training_curves.png  |  results.json")

Plot saved → /kaggle/working/lightgcn_single_outputs/training_curves.png
FINAL RESULTS  (best checkpoint)
  Best Epoch    : 95
  Recall@20 : 0.1411
  NDCG@20   : 0.1186

Saved to /kaggle/working/lightgcn_single_outputs:
  best_model.pt  |  ckpt_epoch*.pt  |  training_curves.png  |  results.json
